<a href="https://colab.research.google.com/github/tinemyumi/saude-mental-datasus/blob/main/notebooks/12.modelagens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Fluxo Intermunicipal**

**Objetivo**

**Autor:** Larissa Tinem

# **1. Setup**

In [ ]:
!pip install geobr -q
!pip install pysal -q
!pip install spreg libpysal esda

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.0/338.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 43.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import seaborn as sns
import geopandas as gpd
import geobr
from scipy import stats
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import Normalize
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
import geopandas as gpd

# Carrega geometria dos municípios de SP via geobr
print('Carregando geometria...')
municipios_sp = geobr.read_municipality(code_muni='SP', year=2020)
municipios_sp['cod6'] = municipios_sp['code_muni'].astype(str).str[:6]
print(f'Municípios SP no mapa: {len(municipios_sp)}')

# **2. Carregamento dos dados**

In [ ]:
base_analitica = pd.read_parquet('/content/drive/MyDrive/Dados/base_analitica/base_analitica.parquet')

## **Índice Global de Moran**

In [ ]:
import pysal
from libpysal.weights import Queen
from esda.moran import Moran

# Criar código municipal com 6 dígitos para compatibilizar com a base do SUS
municipios_sp['cod_municipio'] = (
    municipios_sp['code_muni']
    .astype(str)
    .str[:6]
)

# Garantir código municipal padronizado na base analítica
base_analitica['cod_municipio'] = (
    base_analitica['cod_municipio']
    .astype(str)
    .str.zfill(6)
    .str[:6]
)

# Filtrar apenas municípios de SP
base_moran = base_analitica[
    base_analitica['cod_municipio'].str.startswith('35')
].copy()

# Lista de anos
anos = sorted(base_moran['ano'].dropna().unique())

# Lista para armazenar resultados
resultados = []

# Loop para calcular Moran em cada ano
for ano in anos:

    # Filtra a base do ano analisado
    dados_ano = base_moran[
        base_moran['ano'] == ano
    ][['cod_municipio', 'taxa_internacao']].copy()

    # Junta dados da taxa com o mapa municipal
    gdf = municipios_sp.merge(
        dados_ano,
        on='cod_municipio',
        how='left'
    )

    # Municípios sem internação recebem taxa 0
    gdf['taxa_internacao'] = gdf['taxa_internacao'].fillna(0)

    # Remove municípios sem geometria ou taxa inválida
    gdf = gdf.dropna(subset=['geometry', 'taxa_internacao']).copy()

    # Cria matriz de vizinhança por contiguidade
    w = Queen.from_dataframe(gdf)
    w.transform = 'r'

    # Variável analisada
    y = gdf['taxa_internacao'].values

    # Calcula Moran Global
    moran = Moran(y, w, permutations=999)

    # Salva resultado
    resultados.append({
        'ano': ano,
        'I': moran.I,
        'p_valor': moran.p_sim,
        'z_score': moran.z_sim
    })

    print(f'Ano {ano} concluído - Moran I: {moran.I:.4f}, p={moran.p_sim:.4f}')

# Criar dataframe final com resultados
df_moran = pd.DataFrame(resultados)

display(df_moran)

In [ ]:
plt.plot(df_moran['ano'], df_moran['I'], marker='o')
plt.title('Evolução do Moran Global')
plt.xlabel('Ano')
plt.ylabel('Moran I')
plt.show()

# **LISA**

In [ ]:
# ============================================================
# LISA (MORAN LOCAL) - IDENTIFICAÇÃO DE CLUSTERS ESPACIAIS
# ============================================================

from libpysal.weights import Queen
from esda.moran import Moran_Local
import numpy as np

# Escolher ano para análise
ano_lisa = 2025

# Filtrar base
dados_ano = base_analitica[
    base_analitica['ano'] == ano_lisa
][['cod_municipio', 'taxa_internacao']].copy()

# Merge com mapa
gdf = municipios_sp.merge(
    dados_ano,
    on='cod_municipio',
    how='left'
)

# Preencher municípios sem internação com 0
gdf['taxa_internacao'] = gdf['taxa_internacao'].fillna(0)

# Criar matriz espacial (Queen)
w = Queen.from_dataframe(gdf)
w.transform = 'r'

# Variável
y = gdf['taxa_internacao'].values

# Calcular LISA
lisa = Moran_Local(y, w, permutations=999)

# ============================================================
# RESULTADOS
# ============================================================

# Estatística local
gdf['lisa_I'] = lisa.Is

# p-valor
gdf['lisa_p'] = lisa.p_sim

# Quadrantes
gdf['lisa_q'] = lisa.q

# Significância (p < 0.05)
gdf['lisa_sig'] = gdf['lisa_p'] < 0.05

In [ ]:
# ============================================================
# CLASSIFICAÇÃO DOS CLUSTERS
# ============================================================

def classificar_lisa(row):
    if not row['lisa_sig']:
        return 'NS'  # Não significativo

    if row['lisa_q'] == 1:
        return 'HH'  # Alto-Alto
    elif row['lisa_q'] == 2:
        return 'LH'  # Baixo-Alto
    elif row['lisa_q'] == 3:
        return 'LL'  # Baixo-Baixo
    elif row['lisa_q'] == 4:
        return 'HL'  # Alto-Baixo

gdf['lisa_cluster'] = gdf.apply(classificar_lisa, axis=1)

print(gdf['lisa_cluster'].value_counts())

In [ ]:
# =====================================================
# MAPA LIMPO COM MUNICÍPIOS DESTACADOS
# =====================================================

import matplotlib.pyplot as plt

# Municípios que serão destacados no mapa
municipios_destaque = [
    'Itapira', 'Amparo', 'Presidente Prudente', 'São Paulo'
]

# Separar municípios destacados
gdf_destaque = gdf[gdf['name_muni'].isin(municipios_destaque)].copy()

# Criar figura
fig, ax = plt.subplots(figsize=(12, 9))

# Plotar mapa base em cinza claro
gdf.plot(
    ax=ax,
    color='#F2F2F2',
    edgecolor='white',
    linewidth=0.3
)

# Plotar municípios destacados em cor escura
gdf_destaque.plot(
    ax=ax,
    color='#2C3E50',
    edgecolor='black',
    linewidth=0.8
)

# Adicionar rótulos
for idx, row in gdf_destaque.iterrows():
    ponto = row['geometry'].representative_point()

    ax.annotate(
        text=row['name_muni'],
        xy=(ponto.x, ponto.y),
        xytext=(4, 4),
        textcoords='offset points',
        fontsize=9,
        fontweight='bold',
        color='black',
        bbox=dict(
            boxstyle='round,pad=0.25',
            fc='white',
            ec='none',
            alpha=0.8
        )
    )

# Ajustes visuais
ax.set_title(
    'Municípios selecionados no Estado de São Paulo',
    fontsize=14,
    fontweight='bold'
)

ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MAPA LISA COM RÓTULOS DESTACADOS
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Criar cor a partir do cluster lisa
mapa_cores = {
    'HH': '#d7191c',   # vermelho
    'LL': '#2c7bb6',   # azul
    'HL': '#fdae61',   # laranja
    'LH': '#abd9e9',   # azul claro
    'NS': '#d3d3d3'    # não significativo
}

# Criar coluna de cor com base no cluster LISA
gdf['cor'] = gdf['lisa_cluster'].map(mapa_cores).fillna('#d3d3d3')

# Criar figura
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Plotar todos os municípios
gdf.plot(
    color=gdf['cor'],
    linewidth=0.04,
    edgecolor='#f2f2f2',
    ax=ax
)

ax.axis('off')

# Contornar só o limite externo dos grupos significativos
for cluster in ['HH', 'LL', 'HL', 'LH']:
    gdf_cluster = gdf[gdf['lisa_cluster'] == cluster]


# ============================================================
# RÓTULOS
# ============================================================

municipios_destaque = [
    'Itapira', 'Presidente Prudente', 'São Paulo', 'Registro', 'Taubaté', 'Ribeirão Preto', 'São João Da Boa Vista', 'São José Do Rio Preto'
]

gdf_rotulo = gdf[gdf['name_muni'].isin(municipios_destaque)].copy()

for idx, row in gdf_rotulo.iterrows():

    x = row['geometry'].centroid.x
    y = row['geometry'].centroid.y

    ax.text(
        x, y,
        row['name_muni'],
        fontsize=9,
        fontweight='bold',
        color='black',
        ha='center'
    )

# ============================================================
# TÍTULO E LEGENDA
# ============================================================

ax.axis('off')

legenda = [
    mpatches.Patch(color='#d7191c', label='Alto-Alto (HH)'),
    mpatches.Patch(color='#2c7bb6', label='Baixo-Baixo (LL)'),
    mpatches.Patch(color='#fdae61', label='Alto-Baixo (HL)'),
    mpatches.Patch(color='#abd9e9', label='Baixo-Alto (LH)'),
    mpatches.Patch(color='#d3d3d3', label='Não significativo')
]

plt.legend(handles=legenda, loc='lower left', fontsize=10)

plt.show()

In [ ]:
# Resumo dos clusters LISA
resumo_lisa = (
    gdf
    .groupby('lisa_cluster')
    .agg(
        municipios=('name_muni', 'count'),
        taxa_media=('taxa_internacao', 'mean'),
        taxa_mediana=('taxa_internacao', 'median')
    )
    .reset_index()
)

resumo_lisa

# **SAR**

In [ ]:
# ============================================================
# REGRESSÃO ESPACIAL SAR / SPATIAL LAG MODEL
# Objetivo:
# Avaliar se a taxa de internação psiquiátrica municipal
# apresenta dependência espacial, controlando por CAPS e população
# ============================================================

import numpy as np
import pandas as pd
from libpysal.weights import Queen
from spreg import ML_Lag

# ============================================================
# 1. DEFINIR ANO DA ANÁLISE
# ============================================================

ano_sar = 2025

# ============================================================
# 2. PREPARAR BASE ANALÍTICA
# ============================================================

dados_sar = base_analitica[
    base_analitica['ano'] == ano_sar
][[
    'cod_municipio',
    'taxa_internacao',
    'caps_por_100k',
    'populacao'
]].copy()

# Padronizar código municipal com 6 dígitos
dados_sar['cod_municipio'] = (
    dados_sar['cod_municipio']
    .astype(str)
    .str.zfill(6)
    .str[:6]
)

# ============================================================
# 3. PREPARAR BASE GEOGRÁFICA
# ============================================================

municipios_sp['cod_municipio'] = (
    municipios_sp['code_muni']
    .astype(str)
    .str[:6]
)

# Juntar dados analíticos ao mapa municipal
gdf_sar = municipios_sp.merge(
    dados_sar,
    on='cod_municipio',
    how='left'
)

# ============================================================
# 4. TRATAR VALORES AUSENTES
# ============================================================

# Municípios sem registro de internação recebem taxa 0
gdf_sar['taxa_internacao'] = gdf_sar['taxa_internacao'].fillna(0)

# Municípios sem CAPS recebem 0 CAPS por 100 mil habitantes
gdf_sar['caps_por_100k'] = gdf_sar['caps_por_100k'].fillna(0)

# Remover municípios sem população válida ou sem geometria
gdf_sar = gdf_sar.dropna(subset=['geometry', 'populacao']).copy()
gdf_sar = gdf_sar[gdf_sar['populacao'] > 0].copy()

# ============================================================
# 5. CRIAR VARIÁVEL DE CONTROLE POPULACIONAL
# ============================================================

# Log da população para controlar porte municipal
gdf_sar['log_pop'] = np.log(gdf_sar['populacao'])

# ============================================================
# 6. CRIAR MATRIZ DE PESOS ESPACIAIS
# ============================================================

# Contiguidade Queen: vizinhos compartilham fronteira ou vértice
w = Queen.from_dataframe(gdf_sar)

# Normalização por linha
w.transform = 'r'

# ============================================================
# 7. DEFINIR VARIÁVEIS DO MODELO
# ============================================================

# Variável dependente
y = gdf_sar[['taxa_internacao']].values

# Variáveis explicativas
X = gdf_sar[['caps_por_100k', 'log_pop']].values

# Nomes das variáveis
name_y = 'taxa_internacao'
name_x = ['caps_por_100k', 'log_pop']

# ============================================================
# 8. RODAR MODELO SAR / SPATIAL LAG
# ============================================================

modelo_sar = ML_Lag(
    y=y,
    x=X,
    w=w,
    name_y=name_y,
    name_x=name_x,
    name_w='Queen',
    name_ds=f'SP_{ano_sar}'
)

# ============================================================
# 9. EXIBIR RESULTADOS
# ============================================================

print(modelo_sar.summary)
